**เริ่มงานโปรเจค Phase 1**

In [1]:
import torch
print(torch.cuda.is_available())# True
print(torch.cuda.get_device_name(0)) # Tesla T4

True
Tesla T4


Step 1 — ตั้ง environment

สร้าง cell ใหม่แล้ว mount

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("Mount สำเร็จ!")

Mounted at /content/drive
Mount สำเร็จ!


เช็คพื้นที่ Drive ก่อนโหลดอะไร

In [3]:
import shutil
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f"ใช้ไป: {used/1e9 :.1f} GB")
print(f"ยังเหลือ: {free/1e9 :.1f} GB")

ใช้ไป: 50.0 GB
ยังเหลือ: 70.9 GB


ติดตั้ง libraries ที่ต้องใช้
รัน cell นี้ทุกครั้งที่เปิด session ใหม่ (library หายทุก session)

In [4]:
!pip install -q pycocotools mediapipe opencv-python-headless tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.9 MB/s eta 0:00:00


In [5]:
import cv2, mediapipe as mp, numpy as np
print(f"OpenCV : {cv2.__version__}")
print(f"MediaPipe : {mp.__version__}")
print(f"NumPy : {np.__version__}")

OpenCV : 4.13.0
MediaPipe : 0.10.35
NumPy : 2.0.2


ตั้ง path variables ส่วนกลาง
ทำครั้งเดียว ใช้ตลอดทั้ง notebook - อย่า hardcode path ใน cell อื่น

In [6]:
import os

BASE = '/content/drive/MyDrive/exercise_pose'
ANNO = f'{BASE}/data/coco/annotations'
IMGS = f'{BASE}/data/coco/images'
PROC = f'{BASE}/data/processed'
CKPT = f'{BASE}/models'

# สร้าง folders ทั้งหมดในครั้งเดียว
for d in [ANNO, f'{IMGS}/val2017', f'{IMGS}/train2017', PROC, CKPT]:
    os.makedirs(d, exist_ok=True)

print("Folders ready!")
print(f"BASE = {BASE}")

Folders ready!
BASE = /content/drive/MyDrive/exercise_pose


Step 2 — โหลด dataset


โหลด annotations zip (~240 MB)
โหลดเข้า Drive โดยตรงด้วย -P เพื่อไม่ต้องผ่าน /content ก่อน

In [7]:
ANN_ZIP = f'{ANNO}/annotations_trainval2017.zip'

if not os.path.exists(f'{ANNO}/person_keypoints_val2017.json'):
    print("กำลังโหลด annotations ... ")
    !wget -q --show-progress -P {ANNO} http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    print("กำลัง unzip ... ")
    !unzip -q {ANN_ZIP} -d {ANNO}/tmp
    # ย้าย json ออกมา แล้วลบ folder ซ้อน
    !mv {ANNO}/tmp/annotations/*.json {ANNO}/
    !rm -rf {ANNO}/tmp {ANN_ZIP}
    print("Annotations ready!")
else:
    print("มีอยู่แล้ว, skip.")

มีอยู่แล้ว, skip.


In [8]:
files = os.listdir(ANNO)
for f in sorted(files):
    size = os.path.getsize(f'{ANNO}/{f}') / 1e6
    print(f"{f:<45} {size:>6.1f} MB")

# ควรเห็น:
# person_keypoints_train2017.json 470 MB +- สำคัญ
# person_keypoints_val2017.json 11 MB +- สำคัญ
# captions_train2017.json ...

captions_train2017.json                         91.9 MB
captions_val2017.json                            3.9 MB
instances_train2017.json                       469.8 MB
instances_val2017.json                          20.0 MB
person_keypoints_train2017.json                238.9 MB
person_keypoints_val2017.json                   10.0 MB


โหลดและ unzip val images
ทำให้รู้จัก pipeline ก่อน ใช้เวลาโหลดประมาณ 10-20 นาทีตาม internet

In [10]:
VAL_DIR = f'{IMGS}/val2017'

if not os.path.exists(VAL_DIR) or len(os.listdir(VAL_DIR)) < 100:
    print("กำลังโหลด val2017 images (~1GB) ... ")
    !wget -q --show-progress -P {IMGS} http://images.cocodataset.org/zips/val2017.zip
    print("กำลัง unzip ... (อาจใช้เวลา 3-5 นาที)")
    !unzip -q {IMGS}/val2017.zip -d {IMGS}
    !rm {IMGS}/val2017.zip
    print("val2017 ready!")
else:
    print(f"มีอยู่แล้ว: {len(os.listdir(VAL_DIR))} รูป")

กำลังโหลด val2017 images (~1GB) ... 
val2017.zip         100%[===================>] 777.80M  30.4MB/s    in 65s     
กำลัง unzip ... (อาจใช้เวลา 3-5 นาที)
val2017 ready!


In [12]:
val_count = len(os.listdir(f'{IMGS}/val2017'))
print(f"จำนวนรูปใน val2017: {val_count}")
# ถูกต้อง = 5000

# ดูชือไฟล์ตัวอย่าง
import random
samples = random.sample(os.listdir(f'{IMGS}/val2017'), 5)
for s in samples:
    print(s)

จำนวนรูปใน val2017: 5000
000000353518.jpg
000000521717.jpg
000000559099.jpg
000000500613.jpg
000000162543.jpg


In [ ]:
# comment ไว้ก่อน - โหลดตอน pipeline พร้อมแล้ว
# if not os.path.exists(f'{IMGS}/train2017'):
# !wget -q -- show-progress -P {IMGS}
#http://images.cocodataset.org/zips/train2017.zip
#!unzip -q {IMGS}/train2017.zip -d {IMGS}
#!rm {IMGS}/train2017.zip

โหลด COCO และ print สถิติเบื้องต้น
ถ้ารันผ่านโดยไม่ error = environment และ dataset พร้อม 100%

In [14]:
from pycocotools.coco import COCO

coco_val = COCO(f'{ANNO}/person_keypoints_val2017.json')

# สถิติพื้นฐาน
img_ids = coco_val.getImgIds(catIds=coco_val.getCatIds('person'))
ann_ids = coco_val.getAnnIds(catIds=1, iscrowd=False)

print(f"รูปที่มีคน: {len(img_ids):,}")
print(f"annotations ทั้งหมด: {len(ann_ids):,}")
print(f"keypoint names:")
cat = coco_val.loadCats(1)[0]
for i, kp in enumerate(cat['keypoints']):
    print(f" [{i:2d}] {kp}")

loading annotations into memory...
Done (t=1.06s)
creating index...
index created!
รูปที่มีคน: 2,693
annotations ทั้งหมด: 10,777
keypoint names:
 [ 0] nose
 [ 1] left_eye
 [ 2] right_eye
 [ 3] left_ear
 [ 4] right_ear
 [ 5] left_shoulder
 [ 6] right_shoulder
 [ 7] left_elbow
 [ 8] right_elbow
 [ 9] left_wrist
 [10] right_wrist
 [11] left_hip
 [12] right_hip
 [13] left_knee
 [14] right_knee
 [15] left_ankle
 [16] right_ankle
